# Project 1: Trend Following with the 200-Day Moving Average

**Question:** Does it pay off, after costs, to hold the S&P 500 only while it trades above its 200-day moving average?

**Rule:** If SPY closes above its 200-day simple moving average (SMA), hold SPY from the next day on. Otherwise, hold cash.

**Data:** Daily adjusted closing prices of SPY (S&P 500 ETF) from Yahoo Finance, January 1993 to 30 September 2026. The DAX serves as a second market, the 13-week US T-bill yield as the interest rate on cash.

**Assumptions:** 10 basis points of costs per buy or sell. The signal is computed at the close and the position applies from the next day.

**Contents**
1. Data
2. Returns and buy & hold
3. The signal
4. Strategy returns after costs
5. Performance metrics
6. Chart
7. Robustness
8. Tests
9. Conclusion and limitations

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

from backtest import backtest, load_prices, metrics, trades_per_year

END = "2026-09-30"                 # fixed end date: same results on every run
WINDOW = 200                       # length of the moving average in trading days
COSTS = 0.001                      # 10 basis points per buy or sell
COLORS = ["#12302b", "#9fb1aa"]    # strategy (dark green), buy & hold (grey), as on the website

## 1. Data

- **SPY** has traded since January 1993. It is the longest clean history of an investable product on the US market.
- **Adjusted prices** include dividends and stock splits. Without dividends the return would be understated by roughly 2 % per year, which is why the price index `^GSPC` is not used.
- **Local cache and fixed end date:** the data are downloaded once and stored in `data/`. Every run then works with exactly the same data, and an incomplete trading day cannot slip in.

In [ ]:
data = load_prices("SPY", "data/spy.csv", start="1993-01-01", end=END)

print(f"First date:      {data.index[0].date()}")
print(f"Last date:       {data.index[-1].date()}")
print(f"Trading days:    {len(data)}")
print(f"Missing values:  {data.isna().sum().sum()}")
print(f"Duplicate dates: {data.index.duplicated().sum()}")
print(f"Sorted by date:  {data.index.is_monotonic_increasing}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
data["Close"].plot(ax=ax1, title="SPY, linear scale")
data["Close"].plot(ax=ax2, logy=True, title="SPY, log scale")
plt.show()

On the linear scale the years before 2010 look almost flat. On the log scale equal percentage moves cover the same distance, so the crashes of 2000–2002, 2008 and 2020 become comparable. Over long periods the log scale is the honest view.

## 2. Returns and buy & hold

The daily return is today's price divided by yesterday's, minus 1. Wealth grows by multiplying the growth factors (1 + return), not by adding returns. Buy & hold is the benchmark every strategy has to beat.

In [ ]:
data["Return"] = data["Close"].pct_change().fillna(0)
data["Wealth_BH"] = (1 + data["Return"]).cumprod()

# cross-check: compounding the daily returns must give the overall price ratio
print(f"Final value of 1 USD:   {data['Wealth_BH'].iloc[-1]:.4f}")
print(f"Price ratio last/first: {data['Close'].iloc[-1] / data['Close'].iloc[0]:.4f}")

data["Wealth_BH"].plot(logy=True, figsize=(10, 4), color=COLORS[1], title="Buy & hold: value of 1 USD (log scale)")
plt.show()

In [ ]:
print("Largest daily gains:")
print(data["Return"].nlargest(5).map("{:+.1%}".format).to_string())
print("\nLargest daily losses:")
print(data["Return"].nsmallest(5).map("{:+.1%}".format).to_string())

The largest daily moves in both directions cluster in the crises of autumn 2008 and March 2020, plus the rebound on 9 April 2025. Large gains and large losses come together: volatility clusters.

## 3. The signal

- The **200-day SMA** is the average of the last 200 closing prices, recomputed every day.
- The **signal** is 1 if the close is above the SMA, otherwise 0. During the first 199 days there is no average yet, so the signal must be empty (NaN), not 0. Otherwise the strategy would look as if it had been in cash.
- The **position** is the signal shifted by one day. Today's close is only known at the end of the day, so it can only decide tomorrow's return. Using it for today's return would be **look-ahead bias**.
- Trading at the close of the signal day is slightly optimistic. Section 7.3 tests a one-day delay.

In [ ]:
data["SMA"] = data["Close"].rolling(WINDOW).mean()
data["Signal"] = np.where(data["Close"] > data["SMA"], 1.0, 0.0)
data["Signal"] = data["Signal"].where(data["SMA"].notna())   # no signal while the average is missing
data["Position"] = data["Signal"].shift(1)                    # today's signal applies from tomorrow

print(f"Missing SMA values (warm-up): {data['SMA'].isna().sum()}")
print(f"Share of days invested:       {data['Position'].mean():.1%}")

In [ ]:
ax = data.loc["2000":"2010", ["Close", "SMA"]].plot(logy=True, figsize=(10, 4), title="SPY and its 200-day moving average")
ax.legend(["SPY", "200-day SMA"])
plt.show()

## 4. Strategy returns after costs

On every day: when invested, the strategy earns the market return; in cash it earns 0 % (section 7.7 adds interest). Every switch costs 10 basis points, covering the bid-ask spread and fees. Both series start on the same day, the first day with a valid position.

In [ ]:
bt = data[["Return", "Position"]].dropna().copy()
bt["Trade"] = bt["Position"].diff().abs().fillna(bt["Position"].iloc[0])   # entry on day one counts as a buy
bt["Strategy"] = bt["Position"] * bt["Return"] - bt["Trade"] * COSTS
bt["Buy_Hold"] = bt["Return"]

years = (bt.index[-1] - bt.index[0]).days / 365.25
print(f"Period:          {bt.index[0].date()} to {bt.index[-1].date()} ({years:.1f} years)")
print(f"Trades in total: {bt['Trade'].sum():.0f}")
print(f"Trades per year: {trades_per_year(bt):.1f}")
print(f"Cost drag p.a.:  {trades_per_year(bt) * COSTS:.2%}")

# cross-check: the function in backtest.py must give exactly the same result
print(f"Matches backtest(): {np.allclose(backtest(data['Close'], WINDOW, COSTS)['Strategy'], bt['Strategy'])}")

In [ ]:
(1 + bt[["Strategy", "Buy_Hold"]]).cumprod().plot(
    logy=True, figsize=(10, 5), color=COLORS, title="Value of 1 USD after costs (log scale)")
plt.show()

## 5. Performance metrics

| Metric | Meaning |
| --- | --- |
| CAGR | average growth per year, with compounding |
| Volatility | standard deviation of daily returns × √252 |
| Sharpe ratio | mean / standard deviation × √252, here without a risk-free rate |
| Max drawdown | largest loss from the previous peak |

The mean of daily returns scales with time (× 252), the standard deviation with the square root of time (× √252). That is why √252 remains in the Sharpe ratio.

In [ ]:
results = bt[["Strategy", "Buy_Hold"]].apply(metrics)
results.columns = ["200-day SMA", "Buy & Hold"]
results.loc["Trades per Year"] = [trades_per_year(bt), np.nan]
results.loc["Share Invested"] = [bt["Position"].mean(), 1.0]
results.round(3)

**Reading** (check the numbers after every re-run): the 200-day rule is a risk filter, not a return booster. It earns about 2.9 percentage points less per year than buy & hold (8.0 % vs. 10.9 % CAGR) but cuts the maximum drawdown almost in half (–31 % vs. –55 %). Volatility falls from 18.7 % to 12.1 %, so the Sharpe ratio is slightly higher (0.70 vs. 0.65). With about 6.5 trades per year, costs take roughly 0.65 % per year.

## 6. Chart

Value and drawdown together show both sides of the trade-off: how much the strategy gives up in rising markets and how much it saves in falling ones. Shaded areas mark the periods in cash. The chart is also saved for the website.

In [ ]:
wealth = (1 + bt[["Strategy", "Buy_Hold"]]).cumprod()
wealth.columns = ["200-day SMA", "Buy & Hold"]
drawdown = wealth / wealth.cummax().clip(lower=1) - 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
wealth.plot(ax=ax1, logy=True, color=COLORS)
ax1.fill_between(bt.index, 0, 1, where=bt["Position"] == 0, transform=ax1.get_xaxis_transform(),
                 color="#b39256", alpha=0.15, linewidth=0, label="In cash")
ax1.set_title("Value of 1 USD after costs (log scale)")
ax1.legend()
drawdown.plot(ax=ax2, color=COLORS, legend=False)
ax2.yaxis.set_major_formatter(PercentFormatter(1))
ax2.set_title("Drawdown")
fig.tight_layout()

img_dir = Path("../../docs/img")
img_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(img_dir / "01-trendfolge.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading** (compare with the chart): the cash periods cluster in the long bear markets of 2000–2002 and 2008–2009, where the rule avoided a large part of the decline. In 2020 the crash was so fast that the rule exited only after much of the drop and re-entered after the rebound had started. Section 7.5 puts numbers on both cases.

## 7. Robustness

Testing many variants and reporting only the best one is **overfitting**. All variants are therefore shown. A robust result sits on a plateau: neighbouring settings give similar results.

### 7.1 Window length

All variants start on the same day. A 1,000-day window needs about four years of warm-up, and comparing it over a different period would mix up the effect of the rule with the effect of the period.

In [ ]:
windows = [50, 100, 150, 200, 250, 300, 500, 1000]
runs = {w: backtest(data["Close"], window=w, costs=COSTS) for w in windows}
common_start = max(run.index[0] for run in runs.values())   # same period for every variant

rows = {}
for w, run in runs.items():
    run = run.loc[common_start:]
    rows[w] = pd.concat([metrics(run["Strategy"]), pd.Series({"Trades per Year": trades_per_year(run)})])
rows["Buy & Hold"] = metrics(runs[WINDOW].loc[common_start:, "Buy_Hold"])

columns = ["CAGR", "Volatility", "Sharpe Ratio", "Max Drawdown", "Trades per Year"]
print(f"Common start: {common_start.date()}")
pd.DataFrame(rows).T[columns].round(3)

**Reading:** windows between 150 and 300 days form a plateau, with Sharpe ratios of 0.54–0.74 (buy & hold: 0.59) and maximum drawdowns of –23 % to –38 % (buy & hold: –55 %). The 200-day result is therefore not a lucky spike. Short windows trade too often (about 18 times a year at 50 days) and lose through whipsaws. The 300-day window looks best, but choosing it after seeing the results would be overfitting, so the 200-day window chosen up front remains the main specification.

### 7.2 Costs

In [ ]:
cost_levels = {"0 bp": 0.0, "10 bp": 0.001, "25 bp": 0.0025}
pd.DataFrame({name: metrics(backtest(data["Close"], WINDOW, c)["Strategy"])
              for name, c in cost_levels.items()}).round(3)

**Reading** (fill in after running): at 25 bp the CAGR falls to [x] %. Because the rule trades only about 6.5 times a year, costs matter [little / a lot]. Short windows would suffer much more, since they trade up to three times as often.

### 7.3 Execution delay

In [ ]:
pd.DataFrame({f"{d}-day delay": metrics(backtest(data["Close"], WINDOW, COSTS, delay=d)["Strategy"])
              for d in [1, 2]}).round(3)

**Reading** (fill in after running): trading one day later changes the CAGR from [x] % to [y] % and the Sharpe ratio from [x] to [y]. The result is therefore [robust / sensitive] to the execution assumption.

### 7.4 Sub-periods

In [ ]:
periods = {"1993–2007": ("1993", "2007"), "2008–2019": ("2008", "2019"), "2020–2026": ("2020", "2026")}
sub = {}
for name, (a, b) in periods.items():
    m = bt.loc[a:b, ["Strategy", "Buy_Hold"]].apply(metrics)
    sub[(name, "200-day SMA")] = m["Strategy"]
    sub[(name, "Buy & Hold")] = m["Buy_Hold"]
pd.DataFrame(sub).round(3)

**Reading** (fill in after running): the rule [beats / trails] buy & hold in [period] and [period]. The drawdown reduction [holds in every period / disappears in ...]. If the advantage comes from only one period, it depends on that one crisis.

### 7.5 Crises

Total return from the market peak to the trough (S&P 500 closing dates), plus the COVID recovery to the previous high.

In [ ]:
crises = {
    "Dot-com bust (2000–02)":   ("2000-03-24", "2002-10-09"),
    "Financial crisis (2007–09)": ("2007-10-09", "2009-03-09"),
    "COVID crash (2020)":       ("2020-02-19", "2020-03-23"),
    "COVID recovery (2020)":    ("2020-03-24", "2020-08-18"),
    "Rate shock (2022)":        ("2022-01-03", "2022-10-12"),
}


def total_return(r):
    return (1 + r).prod() - 1


crisis_table = pd.DataFrame({name: bt.loc[a:b, ["Strategy", "Buy_Hold"]].apply(total_return)
                             for name, (a, b) in crises.items()}).T
crisis_table.columns = ["200-day SMA", "Buy & Hold"]
crisis_table.style.format("{:+.1%}")

**Reading** (fill in after running): in the slow bear markets of 2000–2002 and 2007–2009 the rule lost [x] % and [y] %, compared with [x] % and [y] % for buy & hold. In the fast COVID crash it helped [little / a lot], and it missed [x] percentage points of the recovery. Trend rules protect against slow declines, not against sudden crashes.

### 7.6 Second market: DAX

The rule was developed on the S&P 500, so the DAX is unseen data, a small out-of-sample test. The DAX is a performance index that already includes dividends.

In [ ]:
dax = load_prices("^GDAXI", "data/dax.csv", start="1988-01-01", end=END)["Close"].dropna()
dax_bt = backtest(dax, WINDOW, COSTS)

dax_results = dax_bt[["Strategy", "Buy_Hold"]].apply(metrics)
dax_results.columns = ["200-day SMA", "Buy & Hold"]
dax_results.loc["Trades per Year"] = [trades_per_year(dax_bt), np.nan]
print(f"DAX period: {dax_bt.index[0].date()} to {dax_bt.index[-1].date()}")
dax_results.round(3)

**Reading** (fill in after running): on the DAX the rule [also / does not] reduce the drawdown ([x] % vs. [y] %) and reaches a Sharpe ratio of [x] vs. [y] for buy & hold. The result [carries over / does not carry over] to a second market.

### 7.7 Interest on cash

So far cash earns 0 %, which understates the strategy. A realistic alternative is short-term US Treasury bills. `^IRX` is the 13-week T-bill yield in percent per year.

In [ ]:
irx = load_prices("^IRX", "data/irx.csv", start="1993-01-01", end=END)["Close"]
cash_daily = irx / 100 / 252        # annual yield in percent -> daily rate

with_cash = backtest(data["Close"], WINDOW, COSTS, cash_rate=cash_daily)
pd.DataFrame({
    "SMA, cash at 0 %": metrics(bt["Strategy"]),
    "SMA, cash in T-bills": metrics(with_cash["Strategy"]),
    "Buy & Hold": metrics(bt["Buy_Hold"]),
}).round(3)

**Reading** (fill in after running): with T-bill interest the CAGR rises from [x] % to [y] %. The gap to buy & hold shrinks to [z] percentage points. Note that the Sharpe ratio here still ignores the risk-free rate, which would favour buy & hold slightly less.

## 8. Tests

`backtest.py` is covered by small tests in `test_backtest.py` that can be checked by hand: drawdown from the starting capital, CAGR when doubling in two years, no look-ahead, costs only when trading, delay of zero rejected, interest on cash, trades per year. Removing the `shift` in `backtest` makes the look-ahead test fail.

In [ ]:
!{sys.executable} -m pytest -q test_backtest.py

## 9. Conclusion

- **Main result:** the 200-day rule is a risk filter. It gives up about 2.9 percentage points of return per year but nearly halves the maximum drawdown (–31 % vs. –55 %).
- **Robust:** the drawdown reduction holds for windows between 150 and 300 days. [Add: costs, delay, sub-periods, DAX.]
- **Not robust / weaker:** [e.g. fast crashes like 2020, specific periods].
- **Statistical caution:** the Sharpe difference (0.70 vs. 0.65) is small and could be noise. The drawdown reduction is the robust finding.
- **Would I use it with real money?** [Your answer in two sentences, e.g. for which investor and why.]

## Limitations

- Cash earns 0 % in the main specification (section 7.7 relaxes this).
- Taxes are ignored, costs are a flat 10 bp per trade.
- Trading at the close of the signal day is slightly optimistic (section 7.3 tests a delay).
- The rule was developed on one market. The DAX is the only out-of-sample check.
- Yahoo Finance is an unofficial data source.
- The Sharpe ratio ignores the risk-free rate.
- A backtest describes the past and is not a forecast.

## Export for the website

The website is in German, so the export translates labels and uses German number formatting. Paste the output into `docs/data/projects.json` under `"kennzahlen"`.

In [ ]:
import json

LABELS_DE = {"CAGR": "Rendite p. a.", "Volatility": "Volatilität p. a.",
             "Sharpe Ratio": "Sharpe Ratio", "Max Drawdown": "Max. Drawdown"}


def german(x, percent=True):
    return (f"{x:.1%}" if percent else f"{x:.2f}").replace(".", ",").replace("%", " %")


table = results.loc[list(LABELS_DE)]
rows_de = [[LABELS_DE[name], *[german(v, name != "Sharpe Ratio") for v in table.loc[name]]]
           for name in table.index]
print(json.dumps({"spalten": ["200-Tage-Linie", "Buy & Hold"], "zeilen": rows_de}, ensure_ascii=False, indent=2))